In [1]:
class Scalar:
    def __init__(self, data, _children = ()):
        self.grad = 0.0
        self.data = data
        self._prev = set(_children)
        self._backward = lambda:None

    def __repr__(self):
        return f"data = {self.data}"
    
    #operations

    def __add__(self, other):
        if isinstance(other, Scalar):
            other = other
        else:
            other = Scalar(other)

        out = Scalar(self.data + other.data, (self, other)) #same scalar object with self and other as child

        def _backward():
            self.grad += 1 * out.grad #df/dx = dz/dx * df/dz
            other.grad += 1 * out.grad
        out._backward = _backward

        return out
    
    def __mul__(self,other):
        if isinstance(other, Scalar):
            other = other
        else:
            other = Scalar(other)

        out = Scalar(self.data * other.data, (self, other))

        def _backward():
            self.grad +=  other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out
    
    def __neg__(self):
        out = self * Scalar(-1) #just using already exisitng multiplication
        return out 
    
    def __sub__(self, other):
        out = self + (-other)
        return out
    
    #backward prop logic
    def backward(self):
        self.grad = 1.0
        visited = set()
        order =[]

        def traverse(i):
            if i not in visited:
                visited.add(i)
                for j in i._prev:
                    traverse(j)
                order.append(i)
        traverse(self)

        for i in reversed(order):
            i._backward()
                
        




In [2]:
x = Scalar(2)
y = Scalar(3)
z = x*y + x - y
z.backward()
print(x.grad, y.grad)


4.0 1.0
